In [2]:
# [CELL 0] SETUP & DATA LOADING FOR ADDITIONAL EXERCISES
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Soliva"
STUDENT_ID = "TUPM-26-4154"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

# Point to the root repo folder (one level up if running inside Lab7_NumPy_Pandas_Additional_Exercises)
cwd = Path.cwd()
WORK_DIR = cwd.parent if cwd.name == "Lab7_NumPy_Pandas_Additional_Exercises" else cwd
DATA_DIR = WORK_DIR / "data"

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0
observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

print("Setup complete. Usable features:", len(usable_indices), "| Random Seed:", RANDOM_SEED)

Setup complete. Usable features: 474 | Random Seed: 264154


In [3]:
# EX1 - STEP 1 & 2: SELECT FEATURES AND MATRIX
rng_ex1 = np.random.default_rng(RANDOM_SEED)
n_select_ex1 = 5 + (SEED_DIGIT % 3)  # 6 for SEED_DIGIT = 4
ex1_feat_idx = rng_ex1.choice(usable_indices, size=n_select_ex1, replace=False)
ex1_features = [feature_names[i] for i in ex1_feat_idx]

X_ex1 = raw_measurements[:, ex1_feat_idx]
nan_mask_ex1 = np.isnan(X_ex1)

# EX1 - STEP 3 & 4: STATS, MAD, AND ROBUST Z-SCORE
missing_pct_ex1 = nan_mask_ex1.mean(axis=0) * 100.0
means_ex1 = np.nanmean(X_ex1, axis=0)
medians_ex1 = np.nanmedian(X_ex1, axis=0)
sds_ex1 = np.nanstd(X_ex1, axis=0, ddof=1)

mad_ex1 = np.nanmedian(np.abs(X_ex1 - medians_ex1), axis=0)
mad_scale = 1.4826 * mad_ex1
safe_mad_scale = np.where(mad_scale > 0, mad_scale, np.inf)
robust_z_ex1 = (X_ex1 - medians_ex1) / safe_mad_scale

# EX1 - STEP 5 & 6: FLAGS, RANKING, VERIFICATION, AND EXPORT
flagged_mask_ex1 = (~nan_mask_ex1) & (np.abs(robust_z_ex1) > 3.5)
flagged_counts_ex1 = flagged_mask_ex1.sum(axis=0)
flagged_pct_ex1 = (flagged_counts_ex1 / len(X_ex1)) * 100.0

quality_df = pd.DataFrame({
    "Feature": ex1_features,
    "Missing_Pct": missing_pct_ex1,
    "Mean": means_ex1,
    "Median": medians_ex1,
    "Sample_SD": sds_ex1,
    "MAD": mad_ex1,
    "Flagged_Count": flagged_counts_ex1,
    "Flagged_Pct": flagged_pct_ex1,
}).sort_values(by="Flagged_Pct", ascending=False).reset_index(drop=True)

ex_dir = WORK_DIR / "Lab7_NumPy_Pandas_Additional_Exercises"
ex_dir.mkdir(exist_ok=True)
quality_df.to_csv(ex_dir / "measurement_quality.csv", index=False)

for j, feat in enumerate(ex1_features):
    col = X_ex1[:, j]
    med = np.nanmedian(col)
    mad_val = np.nanmedian(np.abs(col - med))
    exp_cnt = int(np.sum((~np.isnan(col)) & (np.abs((col - med) / (1.4826 * mad_val)) > 3.5))) if mad_val > 0 else 0
    assert int(quality_df.loc[quality_df["Feature"] == feat, "Flagged_Count"].iloc[0]) == exp_cnt

print("Ex 1 Selected:", ex1_features)
print("Ex 1 Highest Missing %:", f"{quality_df['Missing_Pct'].max():.2f}%")
print("Ex 1 Top Flagged Feature:", quality_df.loc[0, "Feature"])
print("Ex 1 Total Flagged:", int(flagged_mask_ex1.sum()))

Ex 1 Selected: ['Feature_157', 'Feature_309', 'Feature_105', 'Feature_360', 'Feature_136', 'Feature_038']
Ex 1 Highest Missing %: 91.19%
Ex 1 Top Flagged Feature: Feature_360
Ex 1 Total Flagged: 386


In [4]:
# EX2 - STEP 1 TO 3: BALANCED SAMPLE AND FULL-DATA MEDIAN IMPUTATION
rng_ex2 = np.random.default_rng(RANDOM_SEED + 2)
ex2_feat_idx = rng_ex2.choice(usable_indices, size=5, replace=False)
ex2_features = [feature_names[i] for i in ex2_feat_idx]

full_df = pd.DataFrame(raw_measurements, columns=feature_names)
full_df.insert(0, "Example_ID", np.arange(len(full_df)))
full_df["Result"] = labels
full_df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

pass_pool = full_df.index[full_df["Outcome"] == "Pass"].to_numpy()
fail_pool = full_df.index[full_df["Outcome"] == "Fail"].to_numpy()

per_group_n = min(20 + ID_SUM, len(pass_pool), len(fail_pool))
bal_idx = np.concatenate([
    rng_ex2.choice(pass_pool, size=per_group_n, replace=False),
    rng_ex2.choice(fail_pool, size=per_group_n, replace=False),
])
rng_ex2.shuffle(bal_idx)

full_medians_ex2 = full_df[ex2_features].median()
balanced_df = full_df.loc[bal_idx, ["Example_ID", "Result", "Outcome"] + ex2_features].copy()
balanced_df[ex2_features] = balanced_df[ex2_features].fillna(full_medians_ex2)

# EX2 - STEP 4 & 5: GROUPED STATS, SMD, VERIFICATION, AND EXPORT
grouped_ex2 = balanced_df.groupby("Outcome")[ex2_features].agg(["count", "mean", "median", "std"])

comp_rows = []
for feat in ex2_features:
    n_p = grouped_ex2.loc["Pass", (feat, "count")]
    n_f = grouped_ex2.loc["Fail", (feat, "count")]
    m_p = grouped_ex2.loc["Pass", (feat, "mean")]
    m_f = grouped_ex2.loc["Fail", (feat, "mean")]
    sd_p = grouped_ex2.loc["Pass", (feat, "std")]
    sd_f = grouped_ex2.loc["Fail", (feat, "std")]

    abs_diff = abs(m_f - m_p)
    df_denom = n_p + n_f - 2
    pooled_sd = np.sqrt(((n_p - 1) * sd_p**2 + (n_f - 1) * sd_f**2) / df_denom) if df_denom > 0 else 0.0
    smd = (abs_diff / pooled_sd) if pooled_sd > 0 else 0.0

    p_vals = balanced_df.loc[balanced_df["Outcome"] == "Pass", feat].to_numpy()
    f_vals = balanced_df.loc[balanced_df["Outcome"] == "Fail", feat].to_numpy()
    np_psd = np.sqrt(((len(p_vals) - 1) * np.var(p_vals, ddof=1) + (len(f_vals) - 1) * np.var(f_vals, ddof=1)) / (len(p_vals) + len(f_vals) - 2))
    assert np.isclose(pooled_sd, np_psd, equal_nan=True)

    comp_rows.append({
        "Feature": feat,
        "Pass_Mean": m_p,
        "Fail_Mean": m_f,
        "Abs_Mean_Diff": abs_diff,
        "Pooled_SD": pooled_sd,
        "Abs_SMD": smd,
    })

comparison_df = pd.DataFrame(comp_rows).sort_values(by="Abs_SMD", ascending=False).reset_index(drop=True)
balanced_df.to_csv(ex_dir / "balanced_sample.csv", index=False)
comparison_df.to_csv(ex_dir / "feature_comparison.csv", index=False)

print("Ex 2 Balanced Sample Size:", len(balanced_df))
print("Ex 2 Top Feature:", comparison_df.loc[0, "Feature"], "| Abs SMD:", f"{comparison_df.loc[0, 'Abs_SMD']:.4f}")

Ex 2 Balanced Sample Size: 84
Ex 2 Top Feature: Feature_043 | Abs SMD: 0.3216


In [5]:
# EX3 - STEP 1 TO 5: ELIGIBLE FEATURES, BOOTSTRAP, SENSITIVITY, AND EXPORT
n_boot = 200 + 10 * SEED_DIGIT  # 240 for SEED_DIGIT = 4

eligible_ex3 = [
    feature_names[i]
    for i in usable_indices
    if np.sum((~np.isnan(raw_measurements[:, i])) & (labels == -1)) >= 10
    and np.sum((~np.isnan(raw_measurements[:, i])) & (labels == 1)) >= 10
]

rng_ex3 = np.random.default_rng(RANDOM_SEED + 3)
ex3_features = list(rng_ex3.choice(eligible_ex3, size=5, replace=False))

def evaluate_sensitivity(seed_num):
    r_boot = np.random.default_rng(seed_num)
    p_mask = labels == -1
    f_mask = labels == 1
    med_lookup = {f: np.nanmedian(full_df[f].to_numpy()) for f in ex3_features}

    out_rows = []
    for feat in ex3_features:
        p_raw = full_df.loc[p_mask, feat].to_numpy()
        f_raw = full_df.loc[f_mask, feat].to_numpy()

        p_imp = np.where(np.isnan(p_raw), med_lookup[feat], p_raw)
        f_imp = np.where(np.isnan(f_raw), med_lookup[feat], f_raw)

        diff_av = np.nanmean(f_raw) - np.nanmean(p_raw)
        diff_im = np.mean(f_imp) - np.mean(p_imp)

        b_av = np.empty(n_boot)
        b_im = np.empty(n_boot)
        for k in range(n_boot):
            bp1 = r_boot.choice(p_raw, size=len(p_raw), replace=True)
            bf1 = r_boot.choice(f_raw, size=len(f_raw), replace=True)
            b_av[k] = np.nanmean(bf1) - np.nanmean(bp1)

            bp2 = r_boot.choice(p_imp, size=len(p_imp), replace=True)
            bf2 = r_boot.choice(f_imp, size=len(f_imp), replace=True)
            b_im[k] = np.mean(bf2) - np.mean(bp2)

        ci_av_lo, ci_av_hi = np.percentile(b_av, [2.5, 97.5])
        ci_im_lo, ci_im_hi = np.percentile(b_im, [2.5, 97.5])

        out_rows.append({
            "Feature": feat,
            "Diff_Available": diff_av,
            "Boot_MeanDiff_Available": np.nanmean(b_av),
            "CI95_Av_Low": ci_av_lo,
            "CI95_Av_High": ci_av_hi,
            "Diff_Imputed": diff_im,
            "Boot_MeanDiff_Imputed": np.mean(b_im),
            "CI95_Im_Low": ci_im_lo,
            "CI95_Im_High": ci_im_hi,
            "Abs_Change": abs(diff_im - diff_av),
            "Sign_Changed": np.sign(diff_av) != np.sign(diff_im),
        })

    df_out = pd.DataFrame(out_rows)
    df_out["Rank_Available"] = df_out["Diff_Available"].abs().rank(ascending=False, method="min").astype(int)
    df_out["Rank_Imputed"] = df_out["Diff_Imputed"].abs().rank(ascending=False, method="min").astype(int)
    df_out["Rank_Changed"] = df_out["Rank_Available"] != df_out["Rank_Imputed"]
    return df_out

run1 = evaluate_sensitivity(RANDOM_SEED + 100)
run2 = evaluate_sensitivity(RANDOM_SEED + 100)
repro_verified = run1.equals(run2)

run1.to_csv(ex_dir / "robustness_comparison.csv", index=False)

max_delta = run1["Abs_Change"].max()
sign_changed = run1.loc[run1["Sign_Changed"], "Feature"].tolist()
rank_changed = run1.loc[run1["Rank_Changed"], "Feature"].tolist()

report_txt = (
    "SENSITIVITY AND ROBUSTNESS ANALYSIS REPORT\n"
    f"Student: {LAST_NAME} ({STUDENT_ID})\n"
    f"Bootstrap Repetitions: {n_boot}\n"
    f"Selected Features: {', '.join(ex3_features)}\n"
    f"Largest Change in Mean Difference: {max_delta:.4f}\n"
    f"Features with Changed Difference Sign: {', '.join(sign_changed) if sign_changed else 'None'}\n"
    f"Features with Changed Ranking: {', '.join(rank_changed) if rank_changed else 'None'}\n"
    f"Reproducibility Verified: {repro_verified}\n"
)
(ex_dir / "sensitivity_report.txt").write_text(report_txt, encoding="utf-8")

print("Ex 3 Bootstrap Reps:", n_boot, "| Largest Change:", f"{max_delta:.4f}", "| Reproducible:", repro_verified)

Ex 3 Bootstrap Reps: 240 | Largest Change: 0.0077 | Reproducible: True
